In [23]:
# Verify that TensorFlow can detect your GPU
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

TensorFlow version: 2.15.0
Num GPUs Available:  1


In [24]:
import tensorflow as tf
tf.__version__

'2.15.0'

In [25]:
!pip install kagglehub

In [26]:
import kagglehub
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print("Path to dataset files:", path)

Path to dataset files: /Users/alihamza/.cache/kagglehub/datasets/vipoooool/new-plant-diseases-dataset/versions/2


In [27]:
import os
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt

IMG = (224, 224)
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE

In [28]:
# Find the real train/valid folders inside the downloaded dataset
target_train_path = target_valid_path = None
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'valid' in dirs:
        target_train_path = os.path.join(root, 'train')
        target_valid_path = os.path.join(root, 'valid')
        break
assert target_train_path, f"No train/valid folders found inside {path}"
print(target_train_path)

train_ds = tf.keras.utils.image_dataset_from_directory(
    target_train_path, image_size=IMG, batch_size=BATCH,
    label_mode='categorical', shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    target_valid_path, image_size=IMG, batch_size=BATCH,
    label_mode='categorical', shuffle=False)

num_classes = len(train_ds.class_names)
print("Classes:", num_classes)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

/Users/alihamza/.cache/kagglehub/datasets/vipoooool/new-plant-diseases-dataset/versions/2/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
Classes: 38


In [29]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.2),
])

base = InceptionV3(input_shape=IMG + (3,), weights='imagenet', include_top=False)
base.trainable = False

inputs = layers.Input(shape=IMG + (3,))
x = augment(inputs)
x = layers.Rescaling(1./127.5, offset=-1)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = Model(inputs, outputs)

In [30]:
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 224, 224, 3)       0         
                                                                 
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 inception_v3 (Functional)   (None, 5, 5, 2048)        21802784  
                                                                 
 global_average_pooling2d (  (None, 2048)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense_1 (Dense)             (None, 38)                77862     
                                                           

In [31]:
model.compile(
    optimizer=tf.keras.optimizers.legacy.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

In [32]:
checkpoint = ModelCheckpoint('leaf_disease_model.keras', monitor='val_accuracy',
                             save_best_only=True, mode='max', verbose=1)

history = model.fit(train_ds, validation_data=val_ds,
                    epochs=10, callbacks=[checkpoint])

Epoch 1/10


2026-11-04 19:01:10.308270: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-11-04 19:01:10.676283: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.


   1/1099 [..............................] - ETA: 1:21:38 - loss: 3.9768 - accuracy: 0.0469

2026-11-04 19:01:14.005881: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.


   2/1099 [..............................] - ETA: 5:36 - loss: 4.0203 - accuracy: 0.0547   

2026-11-04 19:01:14.310059: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.


... (21,971 outputs left out)

2026-11-04 20:07:01.080522: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.


1098/1099 [============================>.] - ETA: 0s - loss: 0.3072 - accuracy: 0.9128

2026-11-04 20:07:01.394247: I metal_plugin/src/kernels/stateless_random_op.cc:282] Note the GPU implementation does not produce the same series as CPU implementation.


1099/1099 [==============================] - ETA: 0s - loss: 0.3072 - accuracy: 0.9128
Epoch 10: val_accuracy improved from 0.90627 to 0.90969, saving model to leaf_disease_model.keras
1099/1099 [==============================] - 390s 355ms/step - loss: 0.3072 - accuracy: 0.9128 - val_loss: 0.3105 - val_accuracy: 0.9097


In [34]:
model.save('leaf_disease_final.keras')

In [1]:
# Run this cell once per notebook session to load the model
import tensorflow as tf
from pathlib import Path

model = tf.keras.models.load_model(
    "/Users/alihamza/Developer/CODES/LeafDiseaseAI/leaf_disease_final.keras",
    compile=False
)

train_dir = Path(
    "/Users/alihamza/.cache/kagglehub/datasets/vipoooool/new-plant-diseases-dataset/versions/2/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
)
class_names = sorted(folder.name for folder in train_dir.iterdir() if folder.is_dir())
print("Model ready. Now run the next cell and upload a leaf photo.")


2026-11-04 21:04:30.914021: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-11-04 21:04:30.914049: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-11-04 21:04:30.914056: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-11-04 21:04:30.914286: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-04 21:04:30.914776: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model ready. Now run the next cell and upload a leaf photo.


In [3]:
# Upload a photo; the prediction runs when you select the file
import io
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept="image/*", multiple=False, description="Choose leaf photo")
result = widgets.Output()

def predict_uploaded_image(change):
    value = change["new"]
    if not value:
        return
    if isinstance(value, dict):
        file_info = next(iter(value.values()))
    else:
        file_info = value[0]
    image_bytes = bytes(file_info["content"])
    with result:
        result.clear_output(wait=True)
        image = tf.keras.utils.load_img(io.BytesIO(image_bytes), target_size=(224, 224))
        image_array = tf.keras.utils.img_to_array(image)[None, ...]
        probabilities = model.predict(image_array, verbose=0)[0]
        best = int(probabilities.argmax())
        print("Prediction:", class_names[best])
        print(f"Confidence: {probabilities[best]:.1%}")

uploader.observe(predict_uploaded_image, names="value")
display(uploader, result)


FileUpload(value=(), accept='image/*', description='Choose leaf photo')

Output()